# 06 - Fine-tuning de las CNN
**Proyecto 2 - CGIAR Eyes on the Ground**

Este notebook corre en **Google Colab con GPU**. Entrena la red completa (EfficientNet-B0 y ResNet50, preentrenadas en ImageNet) para clasificar las 4 clases de daño, y compara el resultado con los modelos clásicos sobre embeddings del notebook 05.

**Antes de correrlo:**
1. En Drive, la carpeta `Proyecto2-CGIAR` con `images_224.zip` e `imagenes_meta.csv` (la misma del notebook 04).
2. Entorno de ejecución de Colab con GPU.
3. El código de `src/` subido al repo (`finetuning.py` y `embeddings.py`), y la URL del repo en la celda de configuración.

## 1. Cómo se entrena

- **Dos fases.** Primero solo aprende la capa final, con la base congelada (para no dañar los pesos preentrenados con una capa final todavía aleatoria). Después aprende toda la red con una tasa de aprendizaje más baja.
- **Desbalance de clases:** pesos en la función de pérdida, calculados con la frecuencia de cada clase en `train` (el equivalente a `class_weight="balanced"` de los modelos clásicos).
- **Aumento de datos suave:** giro horizontal, rotación de hasta 10°, recorte con zoom leve y cambios pequeños de brillo y contraste. No se modifica el tono ni la saturación porque el color es justo la señal de la sequía, y no se voltea verticalmente porque el horizonte tiene sentido.
- **Parada temprana** según el F1 macro en validación. Se conservan los pesos de la mejor época.
- **Checkpoint en cada época.** Si Colab corta la sesión, al volver a correr la celda continúa donde se quedó.
- **Test** se usa una sola vez, al final, con los mejores pesos.

## 2. Configuración

In [ ]:
import sys
import json
import subprocess
import time
import zipfile
import shutil
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

# ---------- Ajustar estos valores ----------
REPO_URL = "https://github.com/Emadlgg/proyecto2_Data.git"
CARPETA_DRIVE = "Proyecto2-CGIAR"
LIMITE = None                       # un número (por ejemplo 300) para una prueba rápida de todo el flujo

MODELOS = ["efficientnet_b0", "resnet50"]
COLUMNA_CLASE = "clase4"            # experimento principal: Bueno, Maleza, Sequía, Otro daño
BATCH = 64
EPOCAS_CABEZA = 3                   # fase 1: solo la capa final
EPOCAS_RED = 12                     # fase 2: toda la red (máximo; puede parar antes)
LR_CABEZA = 1e-3
LR_RED = 1e-4
PACIENCIA = 4                       # épocas sin mejorar antes de parar
POTENCIA_PESOS = 1.0                # 1 = pesos "balanced"; 0.5 = más suaves
SEMILLA = 42
# --------------------------------------------

if LIMITE:
    EPOCAS_CABEZA, EPOCAS_RED = 1, 1

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    if not Path("/content/repo").exists():
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)

    RAIZ = Path("/content/repo")
    DRIVE = Path("/content/drive/MyDrive") / CARPETA_DRIVE
    CARPETA_IMAGENES = Path("/content/images_224")
    META = DRIVE / "imagenes_meta.csv"
    SALIDA = DRIVE / "finetuning"
    FIGURAS = DRIVE / "figuras"
    WORKERS = 2
else:
    RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
    CARPETA_IMAGENES = RAIZ / "data" / "processed" / "images_224"
    META = RAIZ / "data" / "processed" / "imagenes_meta.csv"
    SALIDA = RAIZ / "data" / "finetuning"
    FIGURAS = RAIZ / "figuras"
    WORKERS = 0

if LIMITE:
    SALIDA = SALIDA / "_prueba"   # las pruebas rápidas no pisan los resultados reales

sys.path.append(str(RAIZ))

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

from src.finetuning import (crear_loaders, pesos_de_clase, entrenar, predecir,
                            calcular_metricas, fijar_semilla)

sns.set_theme(style="whitegrid")

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", dispositivo)
if dispositivo.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
elif EN_COLAB:
    print("Sin GPU: en Colab, ir a Entorno de ejecución -> Cambiar tipo de entorno -> GPU")

## 3. Imágenes y datos

In [ ]:
if EN_COLAB and not CARPETA_IMAGENES.exists():
    t0 = time.time()
    shutil.copy(DRIVE / "images_224.zip", "/content/images_224.zip")
    with zipfile.ZipFile("/content/images_224.zip") as zf:
        zf.extractall(CARPETA_IMAGENES)
    print(f"Copiado y descomprimido en {time.time() - t0:.0f} s")

print("Fotos en la carpeta:", len(list(CARPETA_IMAGENES.glob("*.jpg"))))

In [ ]:
meta = pd.read_csv(META)
meta = meta[meta["ok"].astype(str).str.lower().eq("true")].reset_index(drop=True)
if LIMITE:
    meta = meta.groupby("split", group_keys=False).head(LIMITE).reset_index(drop=True)

faltan = [a for a in meta["archivo_224"] if not (CARPETA_IMAGENES / a).exists()]
assert not faltan, f"Faltan {len(faltan)} fotos en {CARPETA_IMAGENES}"

CLASES = sorted(meta[COLUMNA_CLASE].unique())
print("Clases:", CLASES)
pd.crosstab(meta[COLUMNA_CLASE], meta["split"])[["train", "val", "test"]]

In [ ]:
fijar_semilla(SEMILLA)
loaders = crear_loaders(CARPETA_IMAGENES, meta, COLUMNA_CLASE, CLASES, batch=BATCH, workers=WORKERS)

y_train = loaders["train"].dataset.etiquetas
pesos = pesos_de_clase(y_train, len(CLASES), potencia=POTENCIA_PESOS)
print("Pesos de clase:", {c: round(float(p), 2) for c, p in zip(CLASES, pesos)})

SALIDA.mkdir(parents=True, exist_ok=True)
(SALIDA / "clases.json").write_text(json.dumps(CLASES, ensure_ascii=False))

## 4. Entrenamiento

Cada modelo guarda en su carpeta (`finetuning/<modelo>/` en Drive) el checkpoint, los mejores pesos, el historial y las métricas. Si un modelo ya tiene métricas guardadas, se salta.

In [ ]:
resultados = {}

for nombre in MODELOS:
    carpeta = SALIDA / nombre
    ruta_metricas = carpeta / "metricas.json"

    if ruta_metricas.exists():
        resultados[nombre] = json.loads(ruta_metricas.read_text())
        print(f"{nombre}: ya entrenado, se cargan las métricas guardadas")
        continue

    print(f"\n=== {nombre} ===")
    modelo, historial = entrenar(
        nombre, loaders, CLASES, pesos, dispositivo, carpeta,
        epocas_cabeza=EPOCAS_CABEZA, epocas_red=EPOCAS_RED,
        lr_cabeza=LR_CABEZA, lr_red=LR_RED, paciencia=PACIENCIA, semilla=SEMILLA,
    )

    # Evaluación con los mejores pesos
    y_val, p_val, _ = predecir(modelo, loaders["val"], dispositivo)
    t0 = time.time()
    y_test, p_test, _ = predecir(modelo, loaders["test"], dispositivo)
    t_pred = time.time() - t0

    m_val = calcular_metricas(y_val, p_val, CLASES)
    m_test = calcular_metricas(y_test, p_test, CLASES)

    registro = {
        "modelo": nombre,
        "mejor_epoca": int(historial.loc[historial["f1_macro_val"].idxmax(), "epoca"]),
        "epocas_corridas": int(len(historial)),
        "f1_macro_val": m_val["f1_macro"],
        "val": m_val,
        "test": m_test,
        "tiempo_entrenamiento_s": float(historial["segundos"].sum()),
        "tiempo_prediccion_test_s": float(t_pred),
        "configuracion": {"batch": BATCH, "epocas_cabeza": EPOCAS_CABEZA, "epocas_red": EPOCAS_RED,
                          "lr_cabeza": LR_CABEZA, "lr_red": LR_RED, "paciencia": PACIENCIA,
                          "potencia_pesos": POTENCIA_PESOS, "semilla": SEMILLA},
    }
    historial.to_csv(carpeta / "historial.csv", index=False)
    np.save(carpeta / "real_test.npy", y_test)
    np.save(carpeta / "pred_test.npy", p_test)
    ruta_metricas.write_text(json.dumps(registro, ensure_ascii=False, indent=2))
    resultados[nombre] = registro

    del modelo
    torch.cuda.empty_cache()

## 5. Resultados

In [ ]:
filas = []
for nombre, r in resultados.items():
    fila = {"embedding": "(fine-tuning)", "modelo": nombre,
            "parametros": f"mejor época {r['mejor_epoca']} de {r['epocas_corridas']}",
            "f1_macro_val": r["f1_macro_val"], "f1_macro_test": r["test"]["f1_macro"],
            "balanced_accuracy_test": r["test"]["balanced_accuracy"], "accuracy_test": r["test"]["accuracy"]}
    fila.update({f"f1_{c}": v for c, v in r["test"]["f1_por_clase"].items()})
    fila.update({"tiempo_entrenamiento_s": r["tiempo_entrenamiento_s"],
                 "tiempo_prediccion_s": r["tiempo_prediccion_test_s"]})
    filas.append(fila)

tabla = pd.DataFrame(filas)
tabla.sort_values("f1_macro_test", ascending=False).round(3)

In [ ]:
# Curvas de entrenamiento
fig, axes = plt.subplots(len(resultados), 2, figsize=(11, 3.8 * len(resultados)), squeeze=False)
for fila_ax, nombre in zip(axes, resultados):
    h = pd.read_csv(SALIDA / nombre / "historial.csv")
    fila_ax[0].plot(h["epoca"], h["perdida_train"], marker="o", label="train")
    fila_ax[0].plot(h["epoca"], h["perdida_val"], marker="o", label="val")
    fila_ax[0].set_title(f"{nombre}: pérdida")
    fila_ax[1].plot(h["epoca"], h["f1_macro_val"], marker="o", color="tab:green")
    fila_ax[1].set_title(f"{nombre}: F1 macro en validación")
    cambio = h.loc[h["fase"] == 2, "epoca"].min()
    for ax in fila_ax:
        if pd.notna(cambio):
            ax.axvline(cambio - 0.5, color="gray", linestyle="--", label="inicia la fase 2")
        ax.set_xlabel("época")
    fila_ax[0].legend(fontsize=8)
fig.tight_layout()
FIGURAS.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURAS / "finetuning_curvas.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Matriz de confusión en test, una por modelo
fig, axes = plt.subplots(1, len(resultados), figsize=(6.5 * len(resultados), 5.5), squeeze=False)
for ax, nombre in zip(axes[0], resultados):
    real = np.load(SALIDA / nombre / "real_test.npy")
    pred = np.load(SALIDA / nombre / "pred_test.npy")
    cm = confusion_matrix(real, pred, labels=list(range(len(CLASES))), normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                xticklabels=CLASES, yticklabels=CLASES, cbar=False, ax=ax)
    ax.set_title(f"{nombre} (fila = 100%)")
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Clase real")
fig.tight_layout()
fig.savefig(FIGURAS / "finetuning_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Comparación con los modelos clásicos

Usa `resultados/clasicos_clase4.csv` del repo (salida del notebook 05). Si no está subido al repo, esta celda solo muestra los modelos fine-tuneados.

In [ ]:
ruta_clasicos = RAIZ / "resultados" / "clasicos_clase4.csv"
partes = [tabla]
if ruta_clasicos.exists():
    clasicos = pd.read_csv(ruta_clasicos)
    partes.insert(0, clasicos[clasicos["embedding"] != "-"])
    base_azar = clasicos.loc[clasicos["modelo"] == "baseline_stratified", "f1_macro_test"].iloc[0]
else:
    base_azar = None
    print("No se encontró", ruta_clasicos)

comp = pd.concat(partes, ignore_index=True)
comp["etiqueta"] = np.where(comp["embedding"] == "(fine-tuning)",
                            comp["modelo"] + "\n(fine-tuning)",
                            comp["modelo"] + "\n+ " + comp["embedding"])
comp["tipo"] = np.where(comp["embedding"] == "(fine-tuning)", "fine-tuning", "clásico sobre embeddings")
comp = comp.sort_values("f1_macro_test")

fig, ax = plt.subplots(figsize=(8, 0.55 * len(comp) + 1.5))
sns.barplot(data=comp, y="etiqueta", x="f1_macro_test", hue="tipo", dodge=False, ax=ax)
if base_azar is not None:
    ax.axvline(base_azar, color="gray", linestyle="--", label="al azar (por proporción)")
ax.set_xlabel("F1 macro (test)")
ax.set_ylabel("")
ax.set_title("Clásicos sobre embeddings vs. fine-tuning (4 clases)")
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
fig.tight_layout()
fig.savefig(FIGURAS / "comparacion_clasicos_finetuning.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Guardar la tabla de resultados

In [ ]:
ruta_tabla = SALIDA.parent / "finetuning_clase4.csv" if not LIMITE else SALIDA / "finetuning_clase4_prueba.csv"
tabla.to_csv(ruta_tabla, index=False)
print("Guardado en", ruta_tabla)

## Siguiente paso

En Drive, dentro de `finetuning/`, quedan los pesos de cada modelo (`<modelo>/mejor.pt`), `clases.json` y `finetuning_clase4.csv`. Se bajan a la laptop: los pesos a `models/` (no se suben a Git), y la tabla de resultados a `resultados/`. Con esos pesos se arma la aplicación.